# Semana 29 · ¿Por qué se atraen los imanes?

Este cuaderno acompaña a la sesión de proyecto de la semana 29: campos de cargas y dipolos, el campo magnético de la Tierra y el giro de una partícula cargada.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. El campo de unas cuantas cargas

El campo de una carga puntual $q$ en la posición $(x_0, y_0)$ apunta en la dirección que va de la carga al punto (hacia fuera si es positiva) y tiene tamaño $kq/r^2$. Para varias cargas, sumamos los vectores. Usamos unidades en las que $k = 1$: lo que nos interesa es la forma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def campo(cargas, X, Y):
    """Suma los campos de una lista de cargas (q, x0, y0) en la malla X, Y."""
    Ex, Ey = np.zeros_like(X), np.zeros_like(Y)
    for q, x0, y0 in cargas:
        dx, dy = X - x0, Y - y0
        r3 = np.hypot(dx, dy) ** 3
        Ex += q * dx / r3
        Ey += q * dy / r3
    return Ex, Ey

x = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x, x)
casos = {
    "una carga positiva": [(1, 0, 0)],
    "dos cargas positivas": [(1, -1, 0), (1, 1, 0)],
    "dipolo (+ y −)": [(1, -1, 0), (-1, 1, 0)],
}
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, (nombre, cargas) in zip(axs, casos.items()):
    Ex, Ey = campo(cargas, X, Y)
    E = np.hypot(Ex, Ey)
    ax.streamplot(X, Y, Ex, Ey, color=np.log10(E), cmap="viridis", density=1.3, linewidth=0.8)
    for q, x0, y0 in cargas:
        ax.plot(x0, y0, "o", ms=12, color="red" if q > 0 else "blue")
    ax.set_title(nombre); ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.tight_layout(); plt.show()

El color indica la intensidad (en escala logarítmica). Fíjate en el dipolo: las líneas salen de la carga positiva y vuelven a la negativa. Es la forma de las limaduras de hierro alrededor de un imán. Con dos cargas iguales, en el punto medio el campo se anula: las líneas se apartan.

## 2. El dipolo cae como $1/r^3$

Comparamos, a lo largo del eje, el campo de una carga sola con el de un dipolo. En una gráfica log-log, una ley $1/r^n$ es una recta de pendiente $-n$ (semana 15).

In [ ]:
r = np.logspace(0.5, 2.5, 100)        # de unas 3 a unas 300 unidades
d = 1.0
E_carga = 1 / r**2
E_dipolo = 1 / (r - d / 2) ** 2 - 1 / (r + d / 2) ** 2

plt.figure(figsize=(7, 5))
plt.loglog(r, E_carga, label="carga sola")
plt.loglog(r, E_dipolo, label="dipolo (exacto)")
plt.loglog(r, 2 * d / r**3, "k--", lw=1, label="aproximación 2qd/r³")
plt.xlabel("distancia r"); plt.ylabel("campo"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()

pend_carga = np.polyfit(np.log10(r), np.log10(E_carga), 1)[0]
pend_dipolo = np.polyfit(np.log10(r), np.log10(E_dipolo), 1)[0]
print(f"pendiente de la carga: {pend_carga:.2f}; pendiente del dipolo: {pend_dipolo:.2f}")

## 3. Las líneas del campo magnético de la Tierra

Si el campo terrestre fuera un dipolo perfecto, cada línea de campo en un plano que pasa por el eje cumpliría

$$r = L\,\cos^2\lambda,$$

donde $r$ es la distancia al centro en radios terrestres, $\lambda$ la latitud magnética y $L$ la distancia a la que la línea cruza el ecuador magnético. Una línea toca el suelo ($r = 1$) donde $\cos^2\lambda = 1/L$. Las partículas atrapadas en la línea $L$ bajan a la atmósfera en esa latitud.

In [ ]:
lat = np.radians(np.linspace(-89, 89, 600))
plt.figure(figsize=(9, 7))
for L in [1.5, 2, 3, 4, 6.6, 8, 10]:
    rr = L * np.cos(lat) ** 2
    dentro = rr >= 1
    for signo in [1, -1]:
        plt.plot(signo * rr[dentro] * np.cos(lat[dentro]), rr[dentro] * np.sin(lat[dentro]), "b", lw=0.8)
    lam = np.degrees(np.arccos(1 / np.sqrt(L)))
    print(f"L = {L:4}: la línea toca el suelo a {lam:4.1f}° de latitud magnética")
tierra = plt.Circle((0, 0), 1, color="steelblue")
plt.gca().add_patch(tierra)
plt.gca().set_aspect("equal"); plt.xlim(-10.5, 10.5); plt.ylim(-6, 6)
plt.title("Líneas del campo magnético de la Tierra (dipolo ideal)"); plt.xlabel("radios terrestres")
plt.show()

Las líneas que llegan hasta 6 a 10 radios terrestres, donde la magnetosfera se agita durante las tormentas, tocan el suelo entre unos 65° y 72° de latitud magnética: es el óvalo auroral. En una tormenta fuerte, se perturban líneas más cercanas, que bajan a latitudes menores. La magnetosfera real está deformada por el viento solar: comprimida del lado diurno y estirada en una cola del lado nocturno.

**Ejercicio.** ¿Qué valor de $L$ hace falta para que la aurora llegue a una latitud magnética de 45°, parecida a la de España?

In [ ]:
# Tu código aquí

In [ ]:
# Solución: cos²(45°) = 1/L
print(f"L = {1 / np.cos(np.radians(45))**2:.1f} radios terrestres")

## 4. Una partícula cargada en un campo magnético

Simulamos una partícula con carga $q$ y masa $m$ en un campo magnético uniforme $\vec{B}$ dirigido a lo largo del eje $z$, con la fuerza de Lorentz $\vec{F} = q\,\vec{v}\times\vec{B}$. Usamos el **método de Boris**, el que emplean los físicos de plasmas: gira la velocidad exactamente sin cambiar su tamaño, así que la energía se conserva aunque el paso de tiempo sea grande. (Un método más simple, como el de Euler, haría que la partícula ganase energía y la espiral se abriera poco a poco.)

In [ ]:
def boris(q, m, B, x0, v0, dt, pasos):
    x, v = np.array(x0, float), np.array(v0, float)
    tray = [x.copy()]
    t = q * B * dt / (2 * m)
    s = 2 * t / (1 + t @ t)
    for _ in range(pasos):
        v_prima = v + np.cross(v, t)
        v = v + np.cross(v_prima, s)
        x = x + v * dt
        tray.append(x.copy())
    return np.array(tray), v

q, m = 1.0, 1.0
B = np.array([0.0, 0.0, 1.0])
v0 = np.array([1.0, 0.0, 0.2])          # componente perpendicular 1, paralela 0,2
tray, v_final = boris(q, m, B, [0, 0, 0], v0, dt=0.05, pasos=1000)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.plot(*tray.T, lw=1)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z (dirección del campo)")
ax.set_title("Una partícula cargada gira en hélice alrededor del campo")
plt.show()

radio_medido = (tray[:, 0].max() - tray[:, 0].min()) / 2
print(f"radio medido: {radio_medido:.3f}   fórmula m·v⊥/(qB): {m * 1.0 / (q * 1.0):.3f}")
print(f"rapidez al principio: {np.linalg.norm(v0):.6f}   al final: {np.linalg.norm(v_final):.6f}")

La partícula da vueltas con el radio previsto, $r = m v_\perp/(qB)$, y avanza a lo largo del campo con su velocidad paralela. Su rapidez no cambia: el campo magnético solo tuerce la trayectoria.

## 5. Experimenta

1. Duplica la velocidad perpendicular. ¿Cambia el radio? ¿Cambia el tiempo que tarda en dar una vuelta?
2. Cambia el signo de la carga. ¿Hacia dónde gira ahora? (Los electrones y los protones de una aurora giran en sentidos opuestos.)
3. En el campo de cargas, prueba con un **cuadrupolo**: cuatro cargas alternas en los vértices de un cuadrado. ¿Cómo cae su campo con la distancia?

In [ ]:
# Tu experimento aquí